#### Sample script for reading in TC-DROPS data
Brian Tang

<i> 6 November 2025 </i>

In [ ]:
#Libraries

import re
import numpy as np
import xarray as xr
import pandas as pd
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
from metpy.calc import dewpoint_from_relative_humidity
from metpy.plots import SkewT
from metpy.units import units
import seaborn as sns
sns.set_context("paper",font_scale=1.0)

%matplotlib inline

In [ ]:
#open TC-DROPS dataset (we'll open the data on height levels)
ds_tcdrops = xr.open_dataset('/network/rit/lab/tcdynasty/tang/tcdrops/all_storms_height_dropsonde_dataset_ver1.2_updated.nc')

#print dataset info
ds_tcdrops

In [ ]:
#Specify storm and mission(s) of interest
atcfid = 'AL062014' #ATCF identifier
tcdropsid = 'al_edouard2014' #TCDROPS identifier <basin>_<name><year>
storm = re.search(r'_(\D+)\d+', tcdropsid).group(1) #extract just the storm name

#print all available missions for storm
stormname = np.squeeze(ds_tcdrops['storm_name'].values).astype(str)
ind = np.nonzero(stormname == tcdropsid)[0]
flightids = np.squeeze(ds_tcdrops['flight_id'].values).astype(str)
print(np.unique(flightids[ind]))

#specify missions of interest that you want to extract dropsonde data for
flightid = ['20140914I1', '20140914_hs3_gh'] #can list one mission; group missions together to examine combined dropsondes

In [ ]:
#Open IBTracs to get TC track observations
ds_ibtracs = xr.open_dataset('/network/rit/lab/tcdynasty/tang/ibtracs/IBTrACS.since1980.v04r01.nc')

#determine index of TC
atcf = np.squeeze(ds_ibtracs['usa_atcf_id'][:,0].values).astype(str)
atcfind = np.nonzero(atcf == atcfid)[0][0]
print(atcfind)

#get track information of TC
ds_ibtracs_tc = ds_ibtracs[['usa_lat','usa_lon']].isel(storm = atcfind)

#convert to dataframe
df_ibtracs_tc = ds_ibtracs_tc.to_dataframe().dropna(how='all').reset_index()
df_ibtracs_tc = df_ibtracs_tc.drop(columns=['date_time','usa_lat','usa_lon'])
df_ibtracs_tc['time'] = df_ibtracs_tc['time'].dt.round('min')

#resample to every minute and apply cubic interpolation
df_ibtracs_tc = df_ibtracs_tc.set_index('time').resample('1s').interpolate(method='cubic')
df_ibtracs_tc = df_ibtracs_tc.reset_index()

# calculate storm motion
REarth = 6371.8E3
time_array = (df_ibtracs_tc['time']-df_ibtracs_tc['time'].iloc[0])/np.timedelta64(1,'s')
# zonal motion = d(lon)/dt * cos(lat) * Radius of Earth
df_ibtracs_tc['storm_motion_u'] = np.gradient(np.radians(df_ibtracs_tc.lon.values),time_array.values)*REarth*np.cos(np.radians(df_ibtracs_tc.lat.values))
# meridional motion = d(lat)/dt * Radius of Earth
df_ibtracs_tc['storm_motion_v'] = np.gradient(np.radians(df_ibtracs_tc.lat.values),time_array.values)*REarth
df_ibtracs_tc

In [ ]:
#Extract dropsonde data for specified mission(s) of interest
#subset mission(s) of interest
ind = []
for i in range(len(flightid)):
    ind.append(np.nonzero(flightids == flightid[i])[0])
ind = np.hstack(ind)

#get the launch time for each drop
drops = ds_tcdrops[['year','month','day','hour','minute','second','latitude_initial','longitude_initial',\
                       'lat_dist_initial','lon_dist_initial','storm_name','flight_id']].isel(ndrop = ind)
df_mission = drops.to_dataframe()
df_mission['storm_name'] = df_mission['storm_name'].str.decode("utf-8")
df_mission['flight_id'] = df_mission['flight_id'].str.decode("utf-8")

#convert launch date/time information to datetime
df_mission['launchtime'] = df_mission.apply(lambda row: datetime(int(row['year']),int(row['month']),int(row['day']),\
                                        int(row['hour']),int(row['minute']),int(row['second'])) ,axis=1)

#get all drops for mission
ds_tcdrop = ds_tcdrops[['time_elapsed','longitude','latitude','pressure','height','temperature','wvmr','rh','uwind','vwind','theta_e']].isel(ndrop = ind)

#convert to dataframe and clean up
df_tcdrop = ds_tcdrop.to_dataframe().dropna(how='all').reset_index()

#calculate time of observation
df_tcdrop = pd.merge(df_tcdrop,df_mission['launchtime'],on='ndrop')
df_tcdrop['time'] = df_tcdrop['launchtime'] + pd.to_timedelta(df_tcdrop['time_elapsed'], unit='s')
df_tcdrop = df_tcdrop.drop(columns=['time_elapsed','nlevels','launchtime'])

#merge with resampled IBTracs data
df_tcdrop['time'] = df_tcdrop['time'].dt.round('s') #round time in TC-DROPS to nearest second
df_tcdrop = pd.merge(df_tcdrop,df_ibtracs_tc,on='time')

#rename TC position columns to avoid confusion
df_tcdrop = df_tcdrop.rename(columns={'lat': 'tclat', 'lon': 'tclon'})

df_tcdrop

In [ ]:
#Perform calculations to transform variables into cylindrical grid and storm-relative frame of reference
#calculate x and y distance relative to center, along with radius and azimuth
tcphi = df_tcdrop.tclat*np.pi/180
tclam = df_tcdrop.tclon*np.pi/180
phi = df_tcdrop.latitude*np.pi/180
lam = df_tcdrop.longitude*np.pi/180

#transformation variables to account for sphericity of lat/lon grid, following Ahern and Cowan (2018)
a = np.clip(np.sin(tcphi)*np.sin(phi)+np.cos(tcphi)*np.cos(phi)*np.cos(phi-tcphi),-1,1)
ctrangle = np.maximum(np.arccos(a),1E-3)
df_tcdrop['x_dist'] = ctrangle*(REarth+df_tcdrop.height)/np.sin(ctrangle)*np.cos(phi)*np.sin(lam-tclam)
df_tcdrop['y_dist'] = ctrangle*(REarth+df_tcdrop.height)/np.sin(ctrangle)*(np.cos(tcphi)*np.sin(phi)-\
                                np.sin(tcphi)*np.cos(phi)*np.cos(lam-tclam))
df_tcdrop['radius'] = (df_tcdrop.x_dist**2.0+df_tcdrop.y_dist**2.0)**0.5
theta_xy = np.arctan2(df_tcdrop.y_dist,df_tcdrop.x_dist)
df_tcdrop['azimuth'] = np.where(theta_xy>=0,theta_xy,theta_xy+2*np.pi)

In [ ]:
#Calculate storm-relative u, v, radial, and tangential wind, following Ahern and Cowan (2018)
df_tcdrop['u_wind_relative']=df_tcdrop.uwind-df_tcdrop.storm_motion_u
df_tcdrop['v_wind_relative']=df_tcdrop.vwind-df_tcdrop.storm_motion_v

eps = 1.E-3
wspd = np.sqrt(df_tcdrop.u_wind_relative**2+df_tcdrop.v_wind_relative**2)
lamprime = lam + eps*df_tcdrop.u_wind_relative/(wspd*np.cos(phi))
phiprime = phi + eps*df_tcdrop.v_wind_relative/wspd

b = np.clip(np.sin(tcphi)*np.sin(phiprime)+np.cos(tcphi)*np.cos(phiprime)*np.cos(phiprime-tcphi),-1,1)
ctrangleprime = np.maximum(np.arccos(b),1E-3)
xprime = ctrangleprime*(REarth+df_tcdrop.height)/np.sin(ctrangleprime)*np.cos(phiprime)*np.sin(lamprime-tclam)
yprime = ctrangleprime*(REarth+df_tcdrop.height)/np.sin(ctrangleprime)*(np.cos(tcphi)*np.sin(phiprime)-\
                        np.sin(tcphi)*np.cos(phiprime)*np.cos(lamprime-tclam))
alpha = np.arctan2(yprime-df_tcdrop.y_dist,xprime-df_tcdrop.x_dist)
alpha = np.where(alpha>=0,alpha,alpha+2*np.pi)

df_tcdrop['radwind_relative'] = wspd*np.cos(df_tcdrop.azimuth-alpha)
df_tcdrop['tanwind_relative'] = wspd*np.cos(df_tcdrop.azimuth-alpha+np.pi/2)

In [ ]:
#Get rid of NaNs and sort
df_tcdrop = df_tcdrop.set_index('time').groupby('ndrop', group_keys=True).apply(lambda group : group.interpolate(method='time',limit_area='inside'))
df_tcdrop = df_tcdrop.drop(columns=['ndrop'])
df_tcdrop = df_tcdrop.dropna().sort_values(['ndrop','time'],ascending=[True,False]).reset_index()

df_tcdrop

#### Examples diagnostics and visualizations

In [ ]:
#Plot dropsonde points in TC-relative frame of reference
#group into drops from low-level and high-level aircraft
pltlim = 400
df_tcdrop_release = df_tcdrop.groupby('ndrop').tail(1)
df_tcdrop_release_hi = df_tcdrop_release[df_tcdrop_release['pressure']<500] #low-level recon drops
df_tcdrop_release_lo = df_tcdrop_release[df_tcdrop_release['pressure']>=500] #high-level recon drops

#plot release points (note: only plots points within axes range)
fig = plt.figure(figsize=(8,8))
plt.plot(df_tcdrop_release_hi['x_dist']/1E3,df_tcdrop_release_hi['y_dist']/1E3,'x',color='red',ms=7,mew=1.5)
plt.plot(df_tcdrop_release_lo['x_dist']/1E3,df_tcdrop_release_lo['y_dist']/1E3,'+',color='blue',ms=8,mew=1.5)
for j in range(len(df_tcdrop_release)):
    plt.text(df_tcdrop_release['x_dist'].iloc[j]/1E3,df_tcdrop_release['y_dist'].iloc[j]/1E3,df_tcdrop_release['ndrop'].iloc[j],clip_on=True)
plt.ylim(-pltlim,pltlim)
plt.xlim(-pltlim,pltlim)

#range rings
crad = np.arange(100,401,100)
for i in range(len(crad)):
    c = plt.Circle((0,0), radius=crad[i], edgecolor='gray', facecolor='none', linestyle='--', linewidth=0.5)
    plt.gca().add_artist(c)
plt.plot([-400,400],[0,0],color='gray',linestyle='--', linewidth=0.5)
plt.plot([0,0],[-400,400],color='gray',linestyle='--', linewidth=0.5)

plt.xlabel('zonal distance from center (km)')
plt.ylabel('meridional distance from center (km)')
plt.show()

In [ ]:
#Plot skew-T and ventilation profile
#select dropsonde to plot
sid = 78 #isolate 

fig = plt.figure(figsize=(8,8))
skew = SkewT(fig, rotation=35, aspect=90)

df_sonde = df_tcdrop[(df_tcdrop['ndrop'] == sid)].sort_values(by=['pressure'])
p = df_sonde['pressure'].values * units.hPa
T = df_sonde['temperature'].values * units.degC
u = df_sonde['uwind'].values * units('m/s')
v = df_sonde['vwind'].values * units('m/s')
Td = dewpoint_from_relative_humidity(df_sonde['temperature'].values * units.degC, df_sonde['rh'].values * units.percent)
skew.plot(p, T, lw=2.5, color='tab:red')
skew.plot(p, Td,lw=2.5, color='tab:green')
skew.plot_barbs(p[::10], u[::10], v[::10], length=6)

skew.ax.set_ylim(1012, 200)
skew.ax.set_xlim(-25, 30)
skew.ax.set_xlabel('Temperature, dewpoint (C)')
skew.ax.set_ylabel('Pressure (hPa)')

skew.plot_dry_adiabats(lw=0.5)
skew.plot_moist_adiabats(lw=0.5)
skew.plot_mixing_lines(lw=0.5)

plt.show()

In [ ]:
#Make a plot of values of some variable averaged over a given layer. Use theta_e as an example.
#determine change of pressure between observations to use as weights in layer average
df_tcdrop['deltap'] = ((df_tcdrop['pressure']+df_tcdrop['pressure'].shift(1))/2) - \
                      ((df_tcdrop['pressure']+df_tcdrop['pressure'].shift(-1))/2)
df_tcdrop['deltap'] = df_tcdrop['deltap'].fillna(0)
df_tcdrop['deltap'] = df_tcdrop['deltap'].mask(df_tcdrop['deltap'] < 0, 0)

#specify layer of interest that you want to average over
df_lay = df_tcdrop.loc[(df_tcdrop['pressure'] >= 900) & (df_tcdrop['pressure'] <= 1000)].groupby('ndrop')

#calculate layer average (add variables to list that you want to average)
df_lay_avg = df_lay.apply(lambda x: pd.Series(np.average(x[['x_dist','y_dist','theta_e']], \
                            weights=x['deltap'], axis=0), ['x_dist','y_dist','theta_e'])).dropna()

#determine radius of azimuth of average locations
theta_xy = np.arctan2(df_lay_avg.y_dist,df_lay_avg.x_dist)
df_lay_avg['radius'] = (df_lay_avg.x_dist**2.0+df_lay_avg.y_dist**2.0)**0.5
df_lay_avg['azimuth'] = np.where(theta_xy>=0,theta_xy,theta_xy+2*np.pi)

In [ ]:
#make plot
pltlim = 400
fig = plt.figure(figsize=(10,8))
h = plt.scatter(df_lay_avg['x_dist']/1E3,df_lay_avg['y_dist']/1E3,c=df_lay_avg['theta_e'],s=150,edgecolors='black',cmap=plt.cm.Spectral_r,vmin=340,vmax=370)
cb = plt.colorbar(h, shrink=0.75)
cb.set_label(r'$\theta_e\;(K)$')
plt.ylim(-pltlim,pltlim)
plt.xlim(-pltlim,pltlim)

crad = np.arange(100,401,100)
for i in range(len(crad)):
    c = plt.Circle((0,0), radius=crad[i], edgecolor='gray', facecolor='none', linestyle='--', linewidth=0.5)
    plt.gca().add_artist(c)
plt.plot([-400,400],[0,0],color='gray',linestyle='--', linewidth=0.5)
plt.plot([0,0],[-400,400],color='gray',linestyle='--', linewidth=0.5)

plt.xlabel('zonal distance from center (km)')
plt.ylabel('meridional distance from center (km)')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from pyproj import Geod
from datetime import timedelta

geod = Geod(ellps='WGS84')

def label_RI_RW(bt_df, hours_between=24, wind_col='vmax_kt', time_col='time'):
    bt = bt_df.sort_values(time_col).set_index(time_col)
    step = int(hours_between / (bt.index.to_series().diff().median().total_seconds() / 3600.0))
    bt['vmax_next'] = bt[wind_col].shift(-step)
    bt['dv'] = bt['vmax_next'] - bt[wind_col]
    RI = bt[bt['dv'] >= 30]
    RW = bt[bt['dv'] <= -30]
    return RI.index.tolist(), RW.index.tolist(), bt

def to_storm_relative(storm_lat, storm_lon, pts_df):
    azs, dists = [], []
    for lat, lon in zip(pts_df['lat'].values, pts_df['lon'].values):
        az1, az2, dist = geod.inv(storm_lon, storm_lat, lon, lat)
        azs.append((az1 + 360) % 360)
        dists.append(dist/1000.0)
    pts = pts_df.copy()
    pts['azimuth_deg'] = azs
    pts['distance_km'] = dists
    return pts

def compute_event_metrics(onset_time, bt_df, flashes_df, outer_rmin=100, outer_rmax=500, pre_hours=24, post_hours=24):
    # find storm position at onset_time (interpolate if necessary)
    storm_row = bt_df.loc[onset_time]
    s_lat, s_lon = storm_row['lat'], storm_row['lon']
    # window
    t0 = pd.to_datetime(onset_time)
    win = (flashes_df['time'] >= t0 - pd.Timedelta(hours=pre_hours)) & (flashes_df['time'] < t0 + pd.Timedelta(hours=post_hours))
    fl_win = flashes_df[win].copy()
    fl_rel = to_storm_relative(s_lat, s_lon, fl_win)
    # outer flashes pre/post
    pre_mask = (fl_rel['time'] >= t0 - pd.Timedelta(hours=pre_hours)) & (fl_rel['time'] < t0)
    post_mask = (fl_rel['time'] >= t0) & (fl_rel['time'] < t0 + pd.Timedelta(hours=post_hours))
    outer_mask = (fl_rel['distance_km'] >= outer_rmin) & (fl_rel['distance_km'] <= outer_rmax)
    outer_pre = fl_rel[pre_mask & outer_mask]
    outer_post = fl_rel[post_mask & outer_mask]
    # counts and rates
    metrics = {
        'outer_pre_count': len(outer_pre),
        'outer_post_count': len(outer_post),
        'outer_pre_rate_per_hr': len(outer_pre) / pre_hours,
        'outer_post_rate_per_hr': len(outer_post) / post_hours,
    }
    # area km^2
    import math
    area = math.pi * (outer_rmax**2 - outer_rmin**2)
    metrics['outer_pre_density_per_km2_hr'] = metrics['outer_pre_rate_per_hr'] / area
    metrics['outer_post_density_per_km2_hr'] = metrics['outer_post_rate_per_hr'] / area
    return metrics


In [ ]:
###########################################          My code here

In [ ]:
import pandas as pd

target_time = pd.to_datetime('2014-09-14 13:00')
time_window = pd.Timedelta(hours=6)

df_focus = df_tcdrop[(df_tcdrop['time'] >= target_time - time_window) &
                     (df_tcdrop['time'] <= target_time + time_window)]




In [ ]:
print(df_tcdrop['time'].min())
print(df_tcdrop['time'].max())


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pltlim = 100  # km, max radius to plot
thin_factor = 10  # only every 10th point

# convert x_dist and y_dist to km
x = df_focus['x_dist']/1E3
y = df_focus['y_dist']/1E3
theta_e = df_focus['theta_e']

# storm-relative winds
u_rel = df_focus['u_wind_relative']
v_rel = df_focus['v_wind_relative']

# mask for radius <= 100 km
mask = df_focus['radius']/1E3 <= pltlim

fig, ax = plt.subplots(figsize=(8,8))
sc = ax.scatter(x[mask], y[mask], c=theta_e[mask], s=100, edgecolors='black', cmap='Spectral_r', vmin=340, vmax=370)

# quiver arrows every 10th point
ax.quiver(x[mask][::thin_factor], y[mask][::thin_factor],
          u_rel[mask][::thin_factor], v_rel[mask][::thin_factor],
          color='k', scale=100)

# range rings (optional)
for r in np.arange(50, pltlim+1, 50):
    circ = plt.Circle((0,0), r, edgecolor='gray', facecolor='none', linestyle='--', linewidth=0.5)
    ax.add_artist(circ)

ax.plot([-pltlim, pltlim],[0,0], color='gray', linestyle='--', linewidth=0.5)
ax.plot([0,0],[-pltlim, pltlim], color='gray', linestyle='--', linewidth=0.5)

ax.set_xlim(-pltlim, pltlim)
ax.set_ylim(-pltlim, pltlim)
ax.set_xlabel('Zonal distance from center (km)')
ax.set_ylabel('Meridional distance from center (km)')
cb = plt.colorbar(sc, ax=ax, shrink=0.75)
cb.set_label('Theta-e (K)')

plt.title('Edouard Dropsondes ~14 Sep 13:00 UTC: Theta-e and Storm-Relative Winds')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pltlim = 400  # max radius to plot (km)
thin_factor = 10  # every 10th point

# x, y distances in km
x = df_focus['x_dist']/1E3
y = df_focus['y_dist']/1E3
theta_e = df_focus['theta_e']

# storm-relative winds
u_rel = df_focus['u_wind_relative']
v_rel = df_focus['v_wind_relative']

# mask for outer-core (>100 km)
mask_outer = df_focus['radius']/1E3 > 100

fig, ax = plt.subplots(figsize=(10,8))

# scatter plot of θe
sc = ax.scatter(x[mask_outer], y[mask_outer], c=theta_e[mask_outer], s=80,
                edgecolors='black', cmap='Spectral_r', vmin=340, vmax=370)

# quiver arrows for storm-relative winds, thinned and smaller
ax.quiver(
    x[mask_outer][::thin_factor], y[mask_outer][::thin_factor],
    u_rel[mask_outer][::thin_factor], v_rel[mask_outer][::thin_factor],
    color='k', scale=200, width=0.002
)

# optional range rings
for r in np.arange(100, pltlim+1, 100):
    circ = plt.Circle((0,0), r, edgecolor='gray', facecolor='none',
                      linestyle='--', linewidth=0.5)
    ax.add_artist(circ)

# cross lines at center
ax.plot([-pltlim, pltlim],[0,0], color='gray', linestyle='--', linewidth=0.5)
ax.plot([0,0],[-pltlim, pltlim], color='gray', linestyle='--', linewidth=0.5)

ax.set_xlim(-pltlim, pltlim)
ax.set_ylim(-pltlim, pltlim)
ax.set_xlabel('Zonal distance from center (km)')
ax.set_ylabel('Meridional distance from center (km)')

# colorbar
cb = plt.colorbar(sc, ax=ax, shrink=0.75)
cb.set_label('Theta-e (K)')

plt.title('Edouard Outer-Core (>100 km) Dropsondes ~14 Sep 13:00 UTC\nTheta-e and Storm-Relative Winds')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
 
# Select outer-core layer (900–700 hPa)
df_outer_layer = df_focus[(df_focus['radius']/1E3 > 100) &
                          (df_focus['pressure'] >= 700) & (df_focus['pressure'] <= 900)].copy()

print(f"Number of points in outer-core 900–700 hPa: {len(df_outer_layer)}")  # check if any data exists

if len(df_outer_layer) > 0:
    # radial bins
    r_bins = np.arange(100, 401, 20)
    r_labels = (r_bins[:-1] + r_bins[1:]) / 2  # midpoints for plotting

    df_outer_layer['r_bin'] = pd.cut(df_outer_layer['radius']/1E3, bins=r_bins, labels=r_labels)

    # groupby with observed=True
    thetae_avg = df_outer_layer.groupby('r_bin', observed=True)['theta_e'].mean()

    # plot
    plt.figure(figsize=(8,6))
    plt.plot(thetae_avg.index.astype(float), thetae_avg.values, marker='o')
    plt.xlabel('Radius (km)')
    plt.ylabel('Mean Theta-e (K)')
    plt.title('Outer-Core Layer-Averaged Theta-e (900–700 hPa)')
    plt.grid(True)
    plt.show()
else:
    print("No data found in the outer-core layer for the specified pressure range.")


In [ ]:
wspd = np.sqrt(u_rel**2 + v_rel**2)
plt.figure(figsize=(8,6))
plt.scatter(df_focus['radius']/1E3, wspd, c=theta_e, cmap='Spectral_r', edgecolors='k')
plt.xlabel('Radius (km)')
plt.ylabel('Storm-Relative Wind (m/s)')
plt.title('Outer-Core Winds and Theta-e')
plt.xlim(100, 400)
plt.colorbar(label='Theta-e (K)')
plt.show()


In [ ]:
import pandas as pd
import numpy as np

# ensure datetime
df_tcdrop['time'] = pd.to_datetime(df_tcdrop['time'])

# --- FILTER TO SEPT 14 ONLY ---
df_14 = df_tcdrop[df_tcdrop['time'].dt.date == pd.to_datetime("2014-09-14").date()].copy()
print(f"Total rows (all levels) on Sept 14: {len(df_14)}")

# --- Identify dropsondes by ndrop ---
unique_sondes = df_14['ndrop'].unique()
print(f"Number of dropsondes on Sept 14: {len(unique_sondes)}\n")

summary = []

for s in unique_sondes:
    df_s = df_14[df_14['ndrop'] == s]

    launch_time = df_s['time'].min()
    end_time = df_s['time'].max()
    n_levels = len(df_s)

    min_p = df_s['pressure'].min()
    max_p = df_s['pressure'].max()

    min_r_km = df_s['radius'].min() / 1000
    max_r_km = df_s['radius'].max() / 1000

    inner_core_flag = min_r_km < 100

    summary.append({
        "ndrop": s,
        "launch_time": launch_time,
        "end_time": end_time,
        "num_levels": n_levels,
        "min_pressure_hPa": np.round(min_p, 1),
        "max_pressure_hPa": np.round(max_p, 1),
        "min_radius_km": np.round(min_r_km, 1),
        "max_radius_km": np.round(max_r_km, 1),
        "went_inside_100km": inner_core_flag
    })

summary_df = pd.DataFrame(summary).sort_values("launch_time")

print("\nDROPPEDSONDE SUMMARY FOR SEPTEMBER 14, 2014\n")
print(summary_df.to_string(index=False))


In [ ]:
# Compute mean profile across all sondes at each pressure level
mean_profile = df_14.groupby('pressure').mean(numeric_only=True)

fig, axs = plt.subplots(1, 3, figsize=(15, 7))

# Temperature
axs[0].plot(mean_profile['temperature'], mean_profile.index)
axs[0].invert_yaxis()
axs[0].set_title("Mean Temperature Profile")

# Moisture
axs[1].plot(mean_profile['rh'], mean_profile.index)
axs[1].invert_yaxis()
axs[1].set_title("Mean RH Profile")

# Theta-e
axs[2].plot(mean_profile['theta_e'], mean_profile.index)
axs[2].invert_yaxis()
axs[2].set_title("Mean θe Profile")

fig.suptitle("Composite Mean Vertical Profiles — Sept 14")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Pressure bins must be increasing!
p_bins = np.arange(250, 951, 20)  # 250 to 950 hPa, 20 hPa bins
p_labels = (p_bins[:-1] + p_bins[1:]) / 2

df_14['p_bin'] = pd.cut(df_14['pressure'], bins=p_bins, labels=p_labels, include_lowest=True)

# Compute smoothed means
smooth = df_14.groupby('p_bin').mean(numeric_only=True)

# Make figure
fig, axs = plt.subplots(1, 3, figsize=(15, 7), sharey=True)

# Temperature
axs[0].plot(smooth['temperature'], smooth.index.astype(float))
axs[0].invert_yaxis()
axs[0].set_title("Temperature (°C)")

# Relative Humidity
axs[1].plot(smooth['rh'], smooth.index.astype(float))
axs[1].invert_yaxis()
axs[1].set_title("RH (%)")

# Theta-e
axs[2].plot(smooth['theta_e'], smooth.index.astype(float))
axs[2].invert_yaxis()
axs[2].set_title("θe (K)")

for ax in axs:
    ax.grid(alpha=0.3)
    ax.set_xlabel("Value")

axs[0].set_ylabel("Pressure (hPa)")
fig.suptitle("Smoothed Composite Vertical Profiles — Sept 14")
plt.show()


In [ ]:
#                  New code thurs apr 9th     

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import pandas as pd

# ── Filter to Sep 14 ───────────────────────────────────────
df_14 = df_tcdrop[df_tcdrop['time'].dt.date == pd.to_datetime('2014-09-14').date()].copy()

# ── Assign quadrants using azimuth (already computed in radians) ──
# Convert azimuth to degrees
df_14['azimuth_deg'] = np.degrees(df_14['azimuth']) % 360

# W quadrant:  135–225° (due west ± 45°)
# NE quadrant: 315–360° and 0–45° (due northeast ± 45°)
def assign_quadrant(az):
    if 135 <= az < 225:
        return 'W'
    elif az >= 315 or az < 45:
        return 'E'
    else:
        return 'Other'

df_14['quadrant'] = df_14['azimuth_deg'].apply(assign_quadrant)

print(df_14['quadrant'].value_counts())

# ── Get release point per drop (tail = lowest pressure = release) ──
df_release = df_14.groupby('ndrop').tail(1).copy()
df_release['quadrant'] = df_release['azimuth_deg'].apply(assign_quadrant)

# ================================================================
# FIGURE 1: Plan-view in storm-relative frame
# ================================================================
pltlim = 400
fig, ax = plt.subplots(figsize=(8, 8))

colors  = {'W': 'red', 'E': 'steelblue', 'Other': 'grey'}
markers = {'W': 'o',   'E': 's',          'Other': '^'}

for quad, group in df_release.groupby('quadrant'):
    ax.scatter(group['x_dist']/1e3, group['y_dist']/1e3,
               color=colors[quad], marker=markers[quad],
               s=120, edgecolors='k', linewidths=0.8,
               label=quad, zorder=4)
    # only label W and NE drops, and clip to axes so nothing prints outside
    if quad != 'Other':
        for _, row in group.iterrows():
            x_km = row['x_dist']/1e3
            y_km = row['y_dist']/1e3
            if abs(x_km) <= pltlim and abs(y_km) <= pltlim:
                ax.text(x_km + 5, y_km + 5, str(int(row['ndrop'])),
                        fontsize=7, color=colors[quad], clip_on=True)

# Range rings
for r in np.arange(100, pltlim+1, 100):
    circ = plt.Circle((0,0), r, edgecolor='gray', facecolor='none',
                      linestyle='--', linewidth=0.7)
    ax.add_artist(circ)
    ax.text(r + 5, 8, f'{r} km', fontsize=7, color='gray')

# Quadrant shading — using x_dist/y_dist directly
# W quadrant:  x < 0, between -45° and +45° from due west
# NE quadrant: x > 0 and y > 0, between 0° and 90° from due north/east
theta_w  = np.radians(np.linspace(135, 225, 200))   # math angles for due west ± 45°
theta_ne = np.radians(np.linspace(45,   90, 200))    # math angles for NE sector (between N and E, upper right)

# W shading: x negative, centered on (-1, 0)
ax.fill(np.concatenate([[0], pltlim * np.cos(theta_w),  [0]]),
        np.concatenate([[0], pltlim * np.sin(theta_w),  [0]]),
        color='red', alpha=0.08, zorder=0)

# E shading: upper-right quadrant
ax.fill(np.concatenate([[0], pltlim * np.cos(theta_e), [0]]),
        np.concatenate([[0], pltlim * np.sin(theta_e), [0]]),
        color='steelblue', alpha=0.08, zorder=0)

ax.plot([-pltlim, pltlim], [0, 0], color='gray', linestyle='--', linewidth=0.5)
ax.plot([0, 0], [-pltlim, pltlim], color='gray', linestyle='--', linewidth=0.5)

ax.set_xlim(-pltlim, pltlim)
ax.set_ylim(-pltlim, pltlim)
ax.set_xlabel('Zonal distance from center (km)', fontsize=12)
ax.set_ylabel('Meridional distance from center (km)', fontsize=12)
ax.legend(title='Quadrant', fontsize=10, loc='upper right')
ax.set_title('Edouard (2014) — Dropsonde Release Locations\nSep 14 | Storm-relative frame', fontsize=12)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('edouard_drops_planview_stormrelative.png', dpi=150, bbox_inches='tight')
plt.show()

# ================================================================
# FIGURE 2: RH profiles W vs NE
# ================================================================
p_bins   = np.arange(200, 1001, 10)
p_labels = (p_bins[:-1] + p_bins[1:]) / 2

fig, axes = plt.subplots(1, 2, figsize=(12, 9), sharey=True, sharex=True)

for ax, quad, color in zip(axes, ['W', 'E'], ['red', 'steelblue']):
    df_q = df_14[df_14['quadrant'] == quad].copy()
    
    # individual drop profiles
    for ndrop, grp in df_q.groupby('ndrop'):
        grp_sorted = grp.sort_values('pressure')
        ax.plot(grp_sorted['rh'], grp_sorted['pressure'],
                color=color, alpha=0.35, linewidth=0.9)
    
    # mean profile (pressure-binned)
    df_q['p_bin'] = pd.cut(df_q['pressure'], bins=p_bins,
                            labels=p_labels, include_lowest=True)
    mean_rh = df_q.groupby('p_bin', observed=True)['rh'].mean()
    
    ax.plot(mean_rh.values, mean_rh.index.astype(float),
            color=color, linewidth=3, label='Quadrant mean',
            path_effects=[pe.Stroke(linewidth=5, foreground='white'),
                          pe.Normal()])
    
    n_drops = df_q['ndrop'].nunique()
    ax.set_title(f'{"W (expect ↑ moisture)" if quad=="W" else "NE (expect ↓ moisture)"}\n'
                 f'n = {n_drops} dropsondes',
                 fontsize=12, fontweight='bold', color=color)
    ax.set_xlim(0, 105)
    ax.set_ylim(1000, 200)
    ax.set_xlabel('Relative Humidity (%)', fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(fontsize=9)

axes[0].set_ylabel('Pressure (hPa)', fontsize=11)
fig.suptitle('Edouard (2014) — Dropsonde RH Profiles by Quadrant\nSep 14',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('edouard_drops_rh_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

# ================================================================
# FIGURE 3: Theta-e profiles W vs NE
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 9), sharey=True, sharex=True)

for ax, quad, color in zip(axes, ['W', 'E'], ['red', 'steelblue']):
    df_q = df_14[df_14['quadrant'] == quad].copy()
    
    for ndrop, grp in df_q.groupby('ndrop'):
        grp_sorted = grp.sort_values('pressure')
        ax.plot(grp_sorted['theta_e'], grp_sorted['pressure'],
                color=color, alpha=0.35, linewidth=0.9)
    
    df_q['p_bin'] = pd.cut(df_q['pressure'], bins=p_bins,
                            labels=p_labels, include_lowest=True)
    mean_te = df_q.groupby('p_bin', observed=True)['theta_e'].mean()
    
    ax.plot(mean_te.values, mean_te.index.astype(float),
            color=color, linewidth=3, label='Quadrant mean',
            path_effects=[pe.Stroke(linewidth=5, foreground='white'),
                          pe.Normal()])
    
    n_drops = df_q['ndrop'].nunique()
    ax.set_title(f'{"W (expect ↑ moisture)" if quad=="W" else "NE (expect ↓ moisture)"}\n'
                 f'n = {n_drops} dropsondes',
                 fontsize=12, fontweight='bold', color=color)
    ax.set_ylim(1000, 200)
    ax.set_xlabel('Equivalent Potential Temperature (K)', fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(fontsize=9)

axes[0].set_ylabel('Pressure (hPa)', fontsize=11)
fig.suptitle('Edouard (2014) — Dropsonde θe Profiles by Quadrant\nSep 14',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('edouard_drops_thetae_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("Time range of df_14:")
print(df_14['time'].min())
print(df_14['time'].max())

print("\nDrops per ndrop with launch time:")
print(df_14.groupby('ndrop')['time'].min().sort_values())

In [ ]:
print(df_mission.reset_index()[['ndrop','flight_id','launchtime']].sort_values('launchtime').to_string())

In [ ]:
# merge flight_id back into df_tcdrop
df_mission_ids = df_mission.reset_index()[['ndrop','flight_id']]
df_tcdrop = df_tcdrop.merge(df_mission_ids, on='ndrop', how='left')

# split by aircraft
df_p3  = df_tcdrop[df_tcdrop['flight_id'] == '20140914I1'].copy()
df_hs3 = df_tcdrop[df_tcdrop['flight_id'] == '20140914_hs3_gh'].copy()

# add quadrant to both
df_p3['azimuth_deg']  = np.degrees(df_p3['azimuth'])  % 360
df_hs3['azimuth_deg'] = np.degrees(df_hs3['azimuth']) % 360
df_p3['quadrant']     = df_p3['azimuth_deg'].apply(assign_quadrant)
df_hs3['quadrant']    = df_hs3['azimuth_deg'].apply(assign_quadrant)

print("P-3 drops:", df_p3['ndrop'].nunique(), 
      "| time:", df_p3['time'].min(), "→", df_p3['time'].max())
print("HS3 drops:", df_hs3['ndrop'].nunique(),
      "| time:", df_hs3['time'].min(), "→", df_hs3['time'].max())
print("\nP-3 quadrants:\n",  df_p3['quadrant'].value_counts())
print("\nHS3 quadrants:\n", df_hs3['quadrant'].value_counts())

In [ ]:
# Changing this to be just E instead of NE 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import pandas as pd

# ── Filter to Sep 14 ───────────────────────────────────────
df_14 = df_tcdrop[df_tcdrop['time'].dt.date == pd.to_datetime('2014-09-14').date()].copy()

# ── Assign quadrants ───────────────────────────────────────
df_14['azimuth_deg'] = np.degrees(df_14['azimuth']) % 360

def assign_quadrant(az):
    if 135 <= az < 225:
        return 'W'
    elif az >= 315 or az < 45:
        return 'E'
    else:
        return 'Other'

df_14['quadrant'] = df_14['azimuth_deg'].apply(assign_quadrant)
print(df_14['quadrant'].value_counts())

df_release = df_14.groupby('ndrop').tail(1).copy()
df_release['quadrant'] = df_release['azimuth_deg'].apply(assign_quadrant)

# ================================================================
# FIGURE 1: Plan-view in storm-relative frame
# ================================================================
pltlim  = 400
colors  = {'W': 'red', 'E': 'steelblue', 'Other': 'grey'}
markers = {'W': 'o',   'E': 's',          'Other': '^'}

fig, ax = plt.subplots(figsize=(8, 8))

for quad, group in df_release.groupby('quadrant'):
    ax.scatter(group['x_dist']/1e3, group['y_dist']/1e3,
               color=colors[quad], marker=markers[quad],
               s=120, edgecolors='k', linewidths=0.8,
               label=quad, zorder=4)
    if quad != 'Other':
        for _, row in group.iterrows():
            x_km = row['x_dist']/1e3
            y_km = row['y_dist']/1e3
            if abs(x_km) <= pltlim and abs(y_km) <= pltlim:
                ax.text(x_km + 5, y_km + 5, str(int(row['ndrop'])),
                        fontsize=7, color=colors[quad], clip_on=True)

# Range rings
for r in np.arange(100, pltlim+1, 100):
    circ = plt.Circle((0,0), r, edgecolor='gray', facecolor='none',
                      linestyle='--', linewidth=0.7)
    ax.add_artist(circ)
    ax.text(r + 5, 8, f'{r} km', fontsize=7, color='gray')

# Quadrant shading
theta_w = np.radians(np.linspace(135, 225, 200))   # due west ± 45°
theta_e = np.radians(np.linspace(-45,  45, 200))   # due east ± 45°

ax.fill(np.concatenate([[0], pltlim * np.cos(theta_w), [0]]),
        np.concatenate([[0], pltlim * np.sin(theta_w), [0]]),
        color='red', alpha=0.08, zorder=0)
ax.fill(np.concatenate([[0], pltlim * np.cos(theta_e), [0]]),
        np.concatenate([[0], pltlim * np.sin(theta_e), [0]]),
        color='steelblue', alpha=0.08, zorder=0)

ax.plot([-pltlim, pltlim], [0, 0], color='gray', linestyle='--', linewidth=0.5)
ax.plot([0, 0], [-pltlim, pltlim], color='gray', linestyle='--', linewidth=0.5)

ax.set_xlim(-pltlim, pltlim)
ax.set_ylim(-pltlim, pltlim)
ax.set_xlabel('Zonal distance from center (km)', fontsize=12)
ax.set_ylabel('Meridional distance from center (km)', fontsize=12)
ax.legend(title='Quadrant', fontsize=10, loc='upper right')
ax.set_title('Edouard (2014) — Dropsonde Release Locations\nSep 14 | Storm-relative frame', fontsize=12)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('edouard_drops_planview_stormrelative.png', dpi=150, bbox_inches='tight')
plt.show()

# ================================================================
# FIGURE 2: RH profiles W vs E
# ================================================================
p_bins   = np.arange(200, 1001, 10)
p_labels = (p_bins[:-1] + p_bins[1:]) / 2

fig, axes = plt.subplots(1, 2, figsize=(12, 9), sharey=True, sharex=True)

for ax, quad, color in zip(axes, ['W', 'E'], ['red', 'steelblue']):
    df_q = df_14[df_14['quadrant'] == quad].copy()

    for ndrop, grp in df_q.groupby('ndrop'):
        grp_sorted = grp.sort_values('pressure')
        ax.plot(grp_sorted['rh'], grp_sorted['pressure'],
                color=color, alpha=0.35, linewidth=0.9)

    df_q['p_bin'] = pd.cut(df_q['pressure'], bins=p_bins,
                            labels=p_labels, include_lowest=True)
    mean_rh = df_q.groupby('p_bin', observed=True)['rh'].mean()
    ax.plot(mean_rh.values, mean_rh.index.astype(float),
            color=color, linewidth=3, label='Quadrant mean',
            path_effects=[pe.Stroke(linewidth=5, foreground='white'),
                          pe.Normal()])

    n_drops = df_q['ndrop'].nunique()
    ax.set_title(f'{"W  (↑ moisture)" if quad=="W" else "E  (↓ moisture)"}\n'
                 f'n = {n_drops} dropsondes',
                 fontsize=12, fontweight='bold', color=color)
    ax.set_xlim(0, 105)
    ax.set_ylim(1000, 200)
    ax.set_xlabel('Relative Humidity (%)', fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(fontsize=9)

axes[0].set_ylabel('Pressure (hPa)', fontsize=11)
fig.suptitle('Edouard (2014) — Dropsonde RH Profiles by Quadrant\nSep 14',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('edouard_drops_rh_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

# ================================================================
# FIGURE 3: Theta-e profiles W vs E
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 9), sharey=True, sharex=True)

for ax, quad, color in zip(axes, ['W', 'E'], ['red', 'steelblue']):
    df_q = df_14[df_14['quadrant'] == quad].copy()

    for ndrop, grp in df_q.groupby('ndrop'):
        grp_sorted = grp.sort_values('pressure')
        ax.plot(grp_sorted['theta_e'], grp_sorted['pressure'],
                color=color, alpha=0.35, linewidth=0.9)

    df_q['p_bin'] = pd.cut(df_q['pressure'], bins=p_bins,
                            labels=p_labels, include_lowest=True)
    mean_te = df_q.groupby('p_bin', observed=True)['theta_e'].mean()
    ax.plot(mean_te.values, mean_te.index.astype(float),
            color=color, linewidth=3, label='Quadrant mean',
            path_effects=[pe.Stroke(linewidth=5, foreground='white'),
                          pe.Normal()])

    n_drops = df_q['ndrop'].nunique()
    ax.set_title(f'{"W  (↑ moisture)" if quad=="W" else "E  (↓ moisture)"}\n'
                 f'n = {n_drops} dropsondes',
                 fontsize=12, fontweight='bold', color=color)
    ax.set_ylim(1000, 200)
    ax.set_xlabel('Equivalent Potential Temperature (K)', fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(fontsize=9)

axes[0].set_ylabel('Pressure (hPa)', fontsize=11)
fig.suptitle('Edouard (2014) — Dropsonde θe Profiles by Quadrant\nSep 14',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('edouard_drops_thetae_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# P-3 RH and Theta-e Vertical Profiles: W vs E
# ================================================================
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

p_bins   = np.arange(200, 1001, 10)
p_labels = (p_bins[:-1] + p_bins[1:]) / 2

# add quadrant to df_p3 using updated assign_quadrant function
df_p3['azimuth_deg'] = np.degrees(df_p3['azimuth']) % 360
df_p3['quadrant']    = df_p3['azimuth_deg'].apply(assign_quadrant)

fig, axes = plt.subplots(1, 4, figsize=(16, 9), sharey=True)

for i, (var, xlabel, xlim) in enumerate([
        ('rh',      'Relative Humidity (%)',         (0, 105)),
        ('theta_e', 'Equivalent Potential Temp (K)', (320, 380))]):
    for j, (quad, color) in enumerate([('W', 'red'), ('E', 'steelblue')]):
        ax = axes[i*2 + j]
        df_q = df_p3[df_p3['quadrant'] == quad].copy()

        # individual drop profiles
        for ndrop, grp in df_q.groupby('ndrop'):
            grp_sorted = grp.sort_values('pressure')
            ax.plot(grp_sorted[var], grp_sorted['pressure'],
                    color=color, alpha=0.4, linewidth=1.0)

        # mean profile
        df_q['p_bin'] = pd.cut(df_q['pressure'], bins=p_bins,
                                labels=p_labels, include_lowest=True)
        mean_var = df_q.groupby('p_bin', observed=True)[var].mean()
        ax.plot(mean_var.values, mean_var.index.astype(float),
                color=color, linewidth=3, label='Mean',
                path_effects=[pe.Stroke(linewidth=5, foreground='white'),
                               pe.Normal()])

        n_drops  = df_q['ndrop'].nunique()
        quad_label = 'W  (↑ moisture)' if quad == 'W' else 'E  (↓ moisture)'
        ax.set_title(f'{quad_label}\nn = {n_drops} sondes',
                     fontsize=11, fontweight='bold', color=color)
        ax.set_xlim(xlim)
        ax.set_ylim(1000, 750)
        ax.set_xlabel(xlabel, fontsize=10)
        ax.grid(True, linestyle='--', alpha=0.4)
        ax.legend(fontsize=9)

axes[0].set_ylabel('Pressure (hPa)', fontsize=11)
fig.suptitle('Edouard (2014) — NOAA P-3 Dropsonde Profiles by Quadrant\n'
             'Sep 14 ~14:57–18:10Z', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('edouard_p3_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# HS3 RH and Theta-e Vertical Profiles: W vs E
# ================================================================

# add quadrant to df_hs3 using updated assign_quadrant function
df_hs3['azimuth_deg'] = np.degrees(df_hs3['azimuth']) % 360
df_hs3['quadrant']    = df_hs3['azimuth_deg'].apply(assign_quadrant)

fig, axes = plt.subplots(1, 4, figsize=(16, 9), sharey=True)

for i, (var, xlabel, xlim) in enumerate([
        ('rh',      'Relative Humidity (%)',         (0, 105)),
        ('theta_e', 'Equivalent Potential Temp (K)', (320, 380))]):
    for j, (quad, color) in enumerate([('W', 'red'), ('E', 'steelblue')]):
        ax = axes[i*2 + j]
        df_q = df_hs3[df_hs3['quadrant'] == quad].copy()

        # individual drop profiles
        for ndrop, grp in df_q.groupby('ndrop'):
            grp_sorted = grp.sort_values('pressure')
            ax.plot(grp_sorted[var], grp_sorted['pressure'],
                    color=color, alpha=0.25, linewidth=0.8)

        # mean profile
        df_q['p_bin'] = pd.cut(df_q['pressure'], bins=p_bins,
                                labels=p_labels, include_lowest=True)
        mean_var = df_q.groupby('p_bin', observed=True)[var].mean()
        ax.plot(mean_var.values, mean_var.index.astype(float),
                color=color, linewidth=3, label='Mean',
                path_effects=[pe.Stroke(linewidth=5, foreground='white'),
                               pe.Normal()])

        n_drops    = df_q['ndrop'].nunique()
        quad_label = 'W  (↑ moisture)' if quad == 'W' else 'E  (↓ moisture)'
        ax.set_title(f'{quad_label}\nn = {n_drops} sondes',
                     fontsize=11, fontweight='bold', color=color)
        ax.set_xlim(xlim)
        ax.set_ylim(1000, 200)
        ax.set_xlabel(xlabel, fontsize=10)
        ax.grid(True, linestyle='--', alpha=0.4)
        ax.legend(fontsize=9)

axes[0].set_ylabel('Pressure (hPa)', fontsize=11)
fig.suptitle('Edouard (2014) — NASA HS3 Global Hawk Dropsonde Profiles by Quadrant\n'
             'Sep 14 13:00Z – Sep 15 08:08Z', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('edouard_hs3_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#####              HARD CODE JUST THE DROPS I WANT

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import pandas as pd

# ─────────────────────────────────────────────────────────────
# FILTER DATE (Sep 14 only)
# ─────────────────────────────────────────────────────────────
df_14 = df_tcdrop[
    df_tcdrop['time'].dt.date == pd.to_datetime('2014-09-14').date()
].copy()


# ─────────────────────────────────────────────────────────────
# SELECT SPECIFIC DROPSONDES
# ─────────────────────────────────────────────────────────────
west_ids = [31, 30, 29]
east_ids = [57, 56, 55]

df_sel = df_14[df_14['ndrop'].isin(west_ids + east_ids)].copy()

def assign_group(ndrop):
    if ndrop in west_ids:
        return "W"
    elif ndrop in east_ids:
        return "E"
    else:
        return "Other"

df_sel['group'] = df_sel['ndrop'].apply(assign_group)

df_release = df_sel.groupby('ndrop').tail(1).copy()


# ================================================================
# FIGURE 1: PLAN VIEW
# ================================================================
pltlim = 400
colors = {'W': 'red', 'E': 'steelblue'}

fig, ax = plt.subplots(figsize=(8, 8))

# ================================================================
# SECTOR HIGHLIGHTING (WEST / EAST)
# ================================================================

pltlim = 400

# angle definitions (storm-relative)
theta_w = np.radians(np.linspace(135, 225, 300))   # W sector
theta_e = np.radians(np.linspace(-45, 45, 300))     # E sector

# West shading
ax.fill(
    np.concatenate([[0], pltlim * np.cos(theta_w), [0]]),
    np.concatenate([[0], pltlim * np.sin(theta_w), [0]]),
    color='red',
    alpha=0.08,
    zorder=0
)

# East shading
ax.fill(
    np.concatenate([[0], pltlim * np.cos(theta_e), [0]]),
    np.concatenate([[0], pltlim * np.sin(theta_e), [0]]),
    color='steelblue',
    alpha=0.08,
    zorder=0
)

for grp, g in df_release.groupby('group'):

    ax.scatter(
        g['x_dist']/1e3,
        g['y_dist']/1e3,
        color=colors[grp],
        marker='o',
        s=140,
        edgecolors='k',
        linewidths=0.8,
        label=grp,
        zorder=4
    )

    # label dropsonde IDs
    # label dropsonde IDs + times
    for _, row in g.iterrows():

        x_km = row['x_dist'] / 1e3
        y_km = row['y_dist'] / 1e3

        tstamp = pd.to_datetime(row['time'])
        time_str = tstamp.strftime("%H:%MZ")

        ax.text(
            x_km + 5,
            y_km + 5,
            f"{int(row['ndrop'])}\n{time_str}",
            fontsize=9,
            color=colors[grp],
            clip_on=True,
            path_effects=[pe.Stroke(linewidth=3, foreground="white"), pe.Normal()]
        )


# Range rings
for r in np.arange(100, pltlim+1, 100):
    circ = plt.Circle((0,0), r, edgecolor='gray', facecolor='none',
                      linestyle='--', linewidth=0.7)
    ax.add_artist(circ)
    ax.text(r + 5, 8, f'{r} km', fontsize=7, color='gray')


ax.plot([-pltlim, pltlim], [0, 0], color='gray', linestyle='--', linewidth=0.5)
ax.plot([0, 0], [-pltlim, pltlim], color='gray', linestyle='--', linewidth=0.5)

ax.set_xlim(-pltlim, pltlim)
ax.set_ylim(-pltlim, pltlim)
ax.set_aspect('equal')

ax.set_xlabel('Zonal distance from center (km)')
ax.set_ylabel('Meridional distance from center (km)')

ax.legend(title='Selected Dropsondes')
ax.set_title('Edouard (2014) — Selected Dropsonde Locations\nSep 14')

plt.tight_layout()
plt.show()


# ================================================================
# FIGURE 2: RH PROFILES
# ================================================================
p_bins = np.arange(200, 1001, 10)
p_labels = (p_bins[:-1] + p_bins[1:]) / 2

fig, axes = plt.subplots(1, 2, figsize=(12, 9), sharey=True, sharex=True)

for ax, grp, color in zip(axes, ['W', 'E'], ['red', 'steelblue']):

    df_g = df_sel[df_sel['group'] == grp].copy()

    for ndrop, g in df_g.groupby('ndrop'):
        g = g.sort_values('pressure')
        ax.plot(g['rh'], g['pressure'],
                color=color, alpha=0.4, linewidth=1)

    df_g['p_bin'] = pd.cut(df_g['pressure'], bins=p_bins,
                            labels=p_labels, include_lowest=True)

    mean_rh = df_g.groupby('p_bin', observed=True)['rh'].mean()

    ax.plot(mean_rh.values, mean_rh.index.astype(float),
            color=color, linewidth=3,
            path_effects=[pe.Stroke(linewidth=5, foreground='white'),
                          pe.Normal()])

    ax.set_title(f"{grp} Dropsondes\nn = {df_g['ndrop'].nunique()}",
                 color=color, fontweight='bold')

    ax.set_xlim(0, 105)
    ax.set_ylim(1000, 200)
    ax.set_xlabel('Relative Humidity (%)')
    ax.grid(True, linestyle='--', alpha=0.4)

axes[0].set_ylabel('Pressure (hPa)')

fig.suptitle('Edouard (2014) — Selected Dropsonde RH Profiles\nSep 14',
             fontweight='bold')

plt.tight_layout()
plt.show()


# ================================================================
# FIGURE 3: THETA-E PROFILES
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 9), sharey=True, sharex=True)

for ax, grp, color in zip(axes, ['W', 'E'], ['red', 'steelblue']):

    df_g = df_sel[df_sel['group'] == grp].copy()

    for ndrop, g in df_g.groupby('ndrop'):
        g = g.sort_values('pressure')
        ax.plot(g['theta_e'], g['pressure'],
                color=color, alpha=0.4, linewidth=1)

    df_g['p_bin'] = pd.cut(df_g['pressure'], bins=p_bins,
                            labels=p_labels, include_lowest=True)

    mean_te = df_g.groupby('p_bin', observed=True)['theta_e'].mean()

    ax.plot(mean_te.values, mean_te.index.astype(float),
            color=color, linewidth=3,
            path_effects=[pe.Stroke(linewidth=5, foreground='white'),
                          pe.Normal()])

    ax.set_title(f"{grp} Dropsondes\nn = {df_g['ndrop'].nunique()}",
                 color=color, fontweight='bold')

    ax.set_ylim(1000, 200)
    ax.set_xlabel('θe (K)')
    ax.grid(True, linestyle='--', alpha=0.4)

axes[0].set_ylabel('Pressure (hPa)')

fig.suptitle('Edouard (2014) — Selected Dropsonde θe Profiles\nSep 14',
             fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# FIGURE: COMBINED RH PROFILES (E vs W + DIFFERENCE SHADING)
# ================================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import pandas as pd

# --- pressure bins ---
p_bins = np.arange(200, 1001, 10)
p_labels = (p_bins[:-1] + p_bins[1:]) / 2

fig, ax = plt.subplots(figsize=(7, 9))

colors = {'W': 'red', 'E': 'steelblue'}
means = {}

# ================================================================
# LOOP THROUGH GROUPS
# ================================================================
for grp in ['W', 'E']:

    df_g = df_sel[df_sel['group'] == grp].copy()

    # --- plot individual sondes (faint) ---
    for ndrop, g in df_g.groupby('ndrop'):
        g = g.sort_values('pressure')
        ax.plot(
            g['rh'], g['pressure'],
            color=colors[grp],
            alpha=0.25,
            linewidth=1
        )

    # --- bin pressure ---
    df_g['p_bin'] = pd.cut(
        df_g['pressure'],
        bins=p_bins,
        labels=p_labels,
        include_lowest=True
    )

    # --- compute mean ---
    mean = df_g.groupby('p_bin', observed=True)['rh'].mean()

    means[grp] = mean

# ================================================================
# ALIGN + SHADE DIFFERENCE
# ================================================================
mean_w = means['W']
mean_e = means['E']

# ensure matching bins
common_index = mean_w.index.intersection(mean_e.index)
mean_w = mean_w.loc[common_index]
mean_e = mean_e.loc[common_index]

p_vals = common_index.astype(float)

# --- SHADE BETWEEN MEANS (THE KEY PART) ---
ax.fill_betweenx(
    p_vals,
    mean_w.values,
    mean_e.values,
    color='gray',
    alpha=0.25,
    label='E–W difference'
)

# ================================================================
# PLOT MEAN PROFILES ON TOP
# ================================================================
ax.plot(
    mean_w.values, p_vals,
    color='red',
    linewidth=3,
    label='W mean',
    path_effects=[pe.Stroke(linewidth=5, foreground='white'), pe.Normal()]
)

ax.plot(
    mean_e.values, p_vals,
    color='steelblue',
    linewidth=3,
    label='E mean',
    path_effects=[pe.Stroke(linewidth=5, foreground='white'), pe.Normal()]
)

# ================================================================
# FORMATTING
# ================================================================
ax.set_xlim(0, 105)
ax.set_ylim(1000, 200)

ax.set_xlabel('Relative Humidity (%)')
ax.set_ylabel('Pressure (hPa)')

ax.grid(True, linestyle='--', alpha=0.4)

ax.legend()

ax.set_title(
    'Edouard (2014) — RH Profiles (E vs W)\nShaded Difference Between Means',
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# FIGURE: COMBINED THETA-E PROFILES (E vs W + DIFFERENCE SHADING)
# ================================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import pandas as pd

# --- pressure bins ---
p_bins = np.arange(200, 1001, 10)
p_labels = (p_bins[:-1] + p_bins[1:]) / 2

fig, ax = plt.subplots(figsize=(7, 9))

colors = {'W': 'red', 'E': 'steelblue'}
means = {}

# ================================================================
# LOOP THROUGH GROUPS
# ================================================================
for grp in ['W', 'E']:

    df_g = df_sel[df_sel['group'] == grp].copy()

    # --- plot individual sondes (faint) ---
    for ndrop, g in df_g.groupby('ndrop'):
        g = g.sort_values('pressure')
        ax.plot(
            g['theta_e'], g['pressure'],
            color=colors[grp],
            alpha=0.25,
            linewidth=1
        )

    # --- bin pressure ---
    df_g['p_bin'] = pd.cut(
        df_g['pressure'],
        bins=p_bins,
        labels=p_labels,
        include_lowest=True
    )

    # --- compute mean ---
    mean = df_g.groupby('p_bin', observed=True)['theta_e'].mean()

    means[grp] = mean

# ================================================================
# ALIGN + SHADE DIFFERENCE
# ================================================================
mean_w = means['W']
mean_e = means['E']

# ensure matching bins
common_index = mean_w.index.intersection(mean_e.index)
mean_w = mean_w.loc[common_index]
mean_e = mean_e.loc[common_index]

p_vals = common_index.astype(float)

# --- SHADE BETWEEN MEANS (KEY FEATURE) ---
ax.fill_betweenx(
    p_vals,
    mean_w.values,
    mean_e.values,
    color='gray',
    alpha=0.25,
    label='E–W difference'
)

# ================================================================
# PLOT MEAN PROFILES ON TOP
# ================================================================
ax.plot(
    mean_w.values, p_vals,
    color='red',
    linewidth=3,
    label='W mean',
    path_effects=[pe.Stroke(linewidth=5, foreground='white'), pe.Normal()]
)

ax.plot(
    mean_e.values, p_vals,
    color='steelblue',
    linewidth=3,
    label='E mean',
    path_effects=[pe.Stroke(linewidth=5, foreground='white'), pe.Normal()]
)

# ================================================================
# FORMATTING
# ================================================================
ax.set_ylim(1000, 200)

ax.set_xlabel('θe (K)')
ax.set_ylabel('Pressure (hPa)')

ax.grid(True, linestyle='--', alpha=0.4)

ax.legend()

ax.set_title(
    'Edouard (2014) — θe Profiles (E vs W)\nShaded Difference Between Means',
    fontweight='bold'
)

plt.tight_layout()
plt.show()